# Домашняя работа: SARSA и Q-learning на FrozenLake

Этот ноутбук закрепляет материалы конспектов по методам TD-control.
Нужно реализовать алгоритмы SARSA и Q-learning, сравнить их поведение и исследовать влияние модификаций среды и типа политики.


## Учебные цели
- Реализовать алгоритмы SARSA и Q-learning для поиска оптимальной политики
- Сравнить on-policy (SARSA) и off-policy (Q-learning) подходы
- Исследовать влияние модификации награды (penalty за падение в озеро)
- Изучить разницу между ε-greedy и softmax политиками
- Сформулировать выводы о применимости каждого метода


## Теоретическая справка

### SARSA (on-policy)
Обновление Q-функции происходит с использованием действия, которое реально выполняется политикой:
$$Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha [r_{t+1} + \gamma Q(s_{t+1}, a_{t+1}) - Q(s_t, a_t)]$$

где $a_{t+1}$ выбирается из текущей политики (например, ε-greedy).

### Q-learning (off-policy)
Обновление Q-функции использует максимальное действие независимо от политики поведения:
$$Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha [r_{t+1} + \gamma \max_a Q(s_{t+1}, a) - Q(s_t, a_t)]$$

### Ключевые различия
- **SARSA**: учитывает риски исследования (действие $a_{t+1}$ может быть случайным из-за ε)
- **Q-learning**: оценивает оптимальную политику напрямую, игнорируя исследование
- **На практике**: SARSA более консервативен, Q-learning более агрессивен


## Как выполнять работу
- Идём сверху вниз; каждый блок с `TODO` нужно заполнить своим кодом
- Если запускаете в Colab, выполните установку зависимостей
- Для воспроизводимости фиксируйте случайные сиды
- В конце заполните секцию с вопросами и выводами


### Подготовка окружения

In [ ]:
# Если работаете в Colab, раскомментируйте строки ниже
# !pip install gymnasium numpy matplotlib tqdm -q

In [ ]:
import random
from dataclasses import dataclass
from typing import Tuple, Callable

import gymnasium as gym
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

In [ ]:
SEED = 2024
WINDOW = 100  # Размер окна для усреднения метрик

random.seed(SEED)
np.random.seed(SEED)

## 1. Модификация среды FrozenLake

Создадим обёртку для FrozenLake, которая добавляет отрицательную награду за падение в озеро.
Это сделает задачу более реалистичной и позволит исследовать разницу между консервативным (SARSA) и агрессивным (Q-learning) поведением.

**Задача:** реализуйте `ModifiedFrozenLakeEnv` — wrapper, который:
- Возвращает `hole_penalty` (например, -1.0) при падении в дыру
- Сохраняет стандартную награду +1 за достижение цели
- Возвращает 0 для обычных шагов


In [ ]:
class ModifiedFrozenLakeEnv(gym.Wrapper):
    """Обёртка для FrozenLake с отрицательной наградой за падение в дыру.
    
    FrozenLake карта 4×4:
        S F F F      S = Start (старт)
        F H F H      F = Frozen (лёд, безопасно)  
        F F F H      H = Hole (дыра, провал)
        H F F G      G = Goal (цель, +1 награда)
    
    Состояния нумеруются 0-15 слева направо, сверху вниз.
    Действия: 0=Left, 1=Down, 2=Right, 3=Up
    """
    
    def __init__(self, env: gym.Env, hole_penalty: float = -1.0):
        """Инициализация wrapper с штрафом за дыры.
        
        TODO: заполните пропуски (замените ... на правильный код)
        """
        # Шаг 1: вызываем конструктор родительского класса
        super().__init__(env)
        
        # Шаг 2: сохраняем штраф за падение в дыру
        self.hole_penalty = ...  # TODO: сохраните hole_penalty
        
        # Шаг 3: получаем карту среды для определения типа клетки
        # env.unwrapped.desc — это numpy-массив с символами b'S', b'F', b'H', b'G'
        self.desc = ...  # TODO: получите описание карты
    
    def step(self, action: int) -> Tuple[int, float, bool, bool, dict]:
        """Выполняет шаг и модифицирует награду при падении в дыру.
        
        TODO: заполните пропуски
        """
        # Шаг 1: выполняем действие в оригинальной среде
        next_state, reward, terminated, truncated, info = self.env.step(action)
        
        # Шаг 2: определяем координаты клетки (row, col) по номеру состояния
        # Формула: state = row * 4 + col, значит row = state // 4, col = state % 4
        row = next_state // 4
        col = ...  # TODO: вычислите номер колонки
        
        # Шаг 3: проверяем, является ли клетка дырой
        # Символ дыры в карте: b'H'
        is_hole = (self.desc[row, col] == b'H')
        
        # Шаг 4: если эпизод завершился в дыре — применяем штраф
        if terminated and is_hole:
            reward = ...  # TODO: замените награду на штраф
            
        return next_state, float(reward), terminated, truncated, info


def make_env(seed: int = SEED, is_slippery: bool = False, hole_penalty: float = 0.0) -> gym.Env:
    """Создаёт FrozenLake с опциональной модификацией награды.
    
    Args:
        seed: сид для воспроизводимости
        is_slippery: True = стохастическая среда (лёд скользкий)
                     False = детерминированная среда (легче для обучения)
        hole_penalty: штраф за падение в дыру (0.0 = стандартная среда)
    """
    env = gym.make("FrozenLake-v1", is_slippery=is_slippery)
    if hole_penalty != 0.0:
        env = ModifiedFrozenLakeEnv(env, hole_penalty=hole_penalty)
    env.reset(seed=seed)
    return env

In [ ]:
# Тест модифицированной среды
# Запускаем несколько эпизодов и проверяем, что штраф за дыры работает

test_env = make_env(hole_penalty=-1.0, is_slippery=False)
penalty_count = 0
total_episodes = 20

for episode in range(total_episodes):
    state, _ = test_env.reset(seed=SEED + 100 + episode)
    done = False
    
    while not done:
        # Случайное действие
        action = test_env.action_space.sample()
        state, reward, terminated, truncated, _ = test_env.step(action)
        done = terminated or truncated
        
        # Проверяем, получили ли мы штраф
        if terminated and reward < 0:
            penalty_count += 1
            break

test_env.close()

print(f"Эпизодов со штрафом: {penalty_count} из {total_episodes}")
print(f"Если penalty_count > 0, значит ModifiedFrozenLakeEnv работает корректно!")

## 2. Политики: ε-greedy и softmax

Реализуем две стратегии исследования:
1. **ε-greedy**: с вероятностью ε выбираем случайное действие, иначе — жадное
2. **Softmax (Boltzmann)**: вероятности пропорциональны $e^{Q(s,a)/\tau}$, где τ — температура


In [ ]:
def epsilon_greedy_action(Q: np.ndarray, state: int, epsilon: float) -> int:
    """Выбор действия по ε-greedy политике.
    
    Логика:
    - С вероятностью ε: случайное действие (exploration)
    - С вероятностью (1-ε): лучшее действие по Q-таблице (exploitation)
    
    TODO: заполните пропуски
    """
    if np.random.random() < epsilon:
        # Исследование: случайное действие из [0, n_actions)
        n_actions = Q.shape[1]  # Количество действий = количество колонок в Q
        return ...  # TODO: верните случайное целое число от 0 до n_actions-1
    else:
        # Эксплуатация: выбираем действие с максимальным Q-значением
        return int(np.argmax(Q[state]))


def softmax_action(Q: np.ndarray, state: int, temperature: float = 1.0) -> int:
    """Выбор действия по softmax (Boltzmann) политике.
    
    Формула вероятности: P(a|s) = exp(Q(s,a)/τ) / Σ exp(Q(s,a')/τ)
    
    - Низкая температура (τ→0): почти жадный выбор
    - Высокая температура (τ→∞): почти равномерный выбор
    
    TODO: заполните пропуски
    """
    # Защита от деления на ноль
    temp = max(temperature, 1e-6)
    
    # Шаг 1: вычисляем логиты = Q-значения / температура
    logits = Q[state] / temp
    
    # Шаг 2: нормализация для численной стабильности (вычитаем максимум)
    # Это предотвращает overflow при exp() больших чисел
    logits = logits - np.max(logits)
    
    # Шаг 3: вычисляем вероятности через softmax
    exp_values = np.exp(logits)
    probs = ...  # TODO: нормализуйте exp_values (делите на сумму)
    
    # Шаг 4: сэмплируем действие согласно вероятностям
    return int(np.random.choice(len(probs), p=probs))

## 3. Реализация SARSA

SARSA — on-policy алгоритм, который обновляет Q-функцию на основе реально выполненных действий:
1. Выбрать $a_t$ из текущей политики (например, ε-greedy)
2. Выполнить $a_t$, получить $(s_{t+1}, r_{t+1})$
3. Выбрать $a_{t+1}$ из той же политики
4. Обновить: $Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha [r_{t+1} + \gamma Q(s_{t+1}, a_{t+1}) - Q(s_t, a_t)]$

**Важно:** действие $a_{t+1}$ выбирается до обновления Q, что делает алгоритм on-policy.


In [ ]:
@dataclass
class SARSAConfig:
    """Конфигурация для алгоритма SARSA."""
    gamma: float = 0.99           # Дисконт-фактор (насколько ценим будущие награды)
    alpha: float = 0.1            # Learning rate (скорость обучения)
    epsilon_start: float = 1.0    # Начальное значение ε (много исследования)
    epsilon_end: float = 0.01     # Финальное значение ε (мало исследования)
    epsilon_decay: float = 0.999  # Множитель затухания ε после каждого эпизода
    num_episodes: int = 10000     # Количество эпизодов обучения
    max_steps: int = 100          # Максимум шагов в одном эпизоде


def sarsa(env: gym.Env, config: SARSAConfig, use_softmax: bool = False, temperature: float = 1.0):
    """Алгоритм SARSA (State-Action-Reward-State-Action).
    
    ON-POLICY алгоритм: обновляем Q используя действие, которое РЕАЛЬНО выполним.
    
    Формула обновления:
        Q(s,a) ← Q(s,a) + α * [r + γ*Q(s',a') - Q(s,a)]
                              ↑ TD-цель    ↑ текущая оценка
        
        Где a' — следующее действие, выбранное ТОЙ ЖЕ политикой!
    
    TODO: заполните пропуски
    
    Returns:
        Q: обученная Q-таблица размера (n_states, n_actions)
        rewards_history: средняя награда за каждые 100 эпизодов
        success_rate_history: доля успешных эпизодов за каждые 100 эпизодов
    """
    n_states = env.observation_space.n   # 16 состояний для FrozenLake 4×4
    n_actions = env.action_space.n       # 4 действия: Left, Down, Right, Up
    
    # Инициализация Q-таблицы нулями (или оптимистично — единицами)
    Q = np.zeros((n_states, n_actions))
    
    # История метрик для графиков
    rewards_history = []
    success_rate_history = []
    
    # Буфер для скользящего среднего
    window_rewards = []
    window_success = []
    
    # Начальное значение epsilon
    epsilon = config.epsilon_start
    
    for episode in tqdm(range(config.num_episodes), desc="SARSA"):
        # === НАЧАЛО ЭПИЗОДА ===
        state, _ = env.reset()
        total_reward = 0.0
        is_success = False
        
        # SARSA: выбираем ПЕРВОЕ действие ДО начала цикла
        if use_softmax:
            action = softmax_action(Q, state, temperature)
        else:
            action = epsilon_greedy_action(Q, state, epsilon)
        
        for step in range(config.max_steps):
            # === ШАГ ВЗАИМОДЕЙСТВИЯ СО СРЕДОЙ ===
            
            # 1. Выполняем действие, получаем новое состояние и награду
            next_state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            
            # 2. Выбираем СЛЕДУЮЩЕЕ действие (on-policy: та же политика!)
            if use_softmax:
                next_action = softmax_action(Q, next_state, temperature)
            else:
                next_action = ...  # TODO: выберите действие по ε-greedy
            
            # === ОБНОВЛЕНИЕ Q-ФУНКЦИИ ===
            
            # 3. Вычисляем TD-цель
            if terminated:
                # Терминальное состояние: нет будущих наград
                td_target = reward
                if reward > 0:
                    is_success = True
            else:
                # Формула SARSA: r + γ * Q(s', a')
                td_target = reward + config.gamma * Q[next_state, next_action]
            
            # 4. Вычисляем TD-ошибку
            td_error = td_target - Q[state, action]
            
            # 5. Обновляем Q-значение
            # Формула: Q(s,a) ← Q(s,a) + α * TD_error
            Q[state, action] += ...  # TODO: примените формулу обновления
            
            # === ПЕРЕХОД К СЛЕДУЮЩЕМУ ШАГУ ===
            state = next_state
            action = next_action  # В SARSA действие уже выбрано!
            
            if terminated or truncated:
                break
        
        # === КОНЕЦ ЭПИЗОДА ===
        
        # Decay epsilon (уменьшаем исследование со временем)
        epsilon = max(config.epsilon_end, epsilon * config.epsilon_decay)
        
        # Сохраняем метрики эпизода
        window_rewards.append(total_reward)
        window_success.append(1.0 if is_success else 0.0)
        
        # Каждые 100 эпизодов сохраняем среднее
        if (episode + 1) % WINDOW == 0:
            rewards_history.append(np.mean(window_rewards))
            success_rate_history.append(np.mean(window_success))
            window_rewards = []
            window_success = []
            
    return Q, rewards_history, success_rate_history

## 4. Реализация Q-learning

Q-learning — off-policy алгоритм, который напрямую оценивает оптимальную Q-функцию:
1. Выбрать $a_t$ из политики поведения (например, ε-greedy)
2. Выполнить $a_t$, получить $(s_{t+1}, r_{t+1})$
3. Обновить: $Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha [r_{t+1} + \gamma \max_a Q(s_{t+1}, a) - Q(s_t, a_t)]$

**Ключевое отличие:** используем $\max_a Q(s_{t+1}, a)$ вместо $Q(s_{t+1}, a_{t+1})$.


In [ ]:
@dataclass
class QLearningConfig:
    """Конфигурация для алгоритма Q-learning."""
    gamma: float = 0.99           # Дисконт-фактор
    alpha: float = 0.1            # Learning rate
    epsilon_start: float = 1.0    # Начальное ε
    epsilon_end: float = 0.01     # Финальное ε
    epsilon_decay: float = 0.999  # Затухание ε
    num_episodes: int = 10000     
    max_steps: int = 100          


def q_learning(env: gym.Env, config: QLearningConfig, use_softmax: bool = False, temperature: float = 1.0):
    """Алгоритм Q-learning.
    
    OFF-POLICY алгоритм: обновляем Q используя ЛУЧШЕЕ возможное действие,
    независимо от того, какое действие мы реально выполним.
    
    Формула обновления:
        Q(s,a) ← Q(s,a) + α * [r + γ*max_a' Q(s',a') - Q(s,a)]
                              ↑ TD-цель (оптимистичная)
        
        Используем max, а не Q(s',a') — это ключевое отличие от SARSA!
    
    TODO: заполните пропуски
    
    Returns:
        Q: обученная Q-таблица
        rewards_history: история средних наград
        success_rate_history: история success rate
    """
    n_states = env.observation_space.n
    n_actions = env.action_space.n
    
    Q = np.zeros((n_states, n_actions))
    
    rewards_history = []
    success_rate_history = []
    window_rewards = []
    window_success = []
    
    epsilon = config.epsilon_start
    
    for episode in tqdm(range(config.num_episodes), desc="Q-learning"):
        # === НАЧАЛО ЭПИЗОДА ===
        state, _ = env.reset()
        total_reward = 0.0
        is_success = False
        
        # Q-learning: НЕ выбираем действие заранее (отличие от SARSA)
        
        for step in range(config.max_steps):
            # === ШАГ ВЗАИМОДЕЙСТВИЯ СО СРЕДОЙ ===
            
            # 1. Выбираем действие (поведенческая политика)
            if use_softmax:
                action = softmax_action(Q, state, temperature)
            else:
                action = epsilon_greedy_action(Q, state, epsilon)
            
            # 2. Выполняем действие
            next_state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            
            # === ОБНОВЛЕНИЕ Q-ФУНКЦИИ ===
            
            # 3. Вычисляем TD-цель (OFF-POLICY: используем max!)
            if terminated:
                td_target = reward
                if reward > 0:
                    is_success = True
            else:
                # Формула Q-learning: r + γ * max_a' Q(s', a')
                # np.max(Q[next_state]) возвращает максимальное Q-значение
                td_target = reward + config.gamma * ...  # TODO: max Q(s', a')
            
            # 4. Обновляем Q-значение
            Q[state, action] += config.alpha * (td_target - Q[state, action])
            
            # === ПЕРЕХОД К СЛЕДУЮЩЕМУ ШАГУ ===
            state = next_state
            # В Q-learning НЕ сохраняем next_action — выберем на следующем шаге
            
            if terminated or truncated:
                break
        
        # === КОНЕЦ ЭПИЗОДА ===
        epsilon = max(config.epsilon_end, epsilon * config.epsilon_decay)
        
        window_rewards.append(total_reward)
        window_success.append(1.0 if is_success else 0.0)
        
        if (episode + 1) % WINDOW == 0:
            rewards_history.append(np.mean(window_rewards))
            success_rate_history.append(np.mean(window_success))
            window_rewards = []
            window_success = []
            
    return Q, rewards_history, success_rate_history

## 5. Эксперимент A: SARSA vs Q-learning на стандартной FrozenLake

Сравним оба алгоритма на стандартной среде (без модификации наград).

### Цель эксперимента
Понять разницу между **on-policy** (SARSA) и **off-policy** (Q-learning) обучением.

### План эксперимента
1. Создать среду `FrozenLake` без штрафа за дыры (`hole_penalty=0.0`)
2. Обучить SARSA с конфигурацией `SARSAConfig()`
3. Обучить Q-learning с конфигурацией `QLearningConfig()`
4. Сравнить:
   - Финальный success rate (% успешных эпизодов)
   - Скорость сходимости (график обучения)
   - Выученные политики (карта действий 4×4)

### Ожидаемые результаты
- Q-learning должен сходиться **быстрее** (оптимистичные обновления через max)
- SARSA может быть более **стабильным** (учитывает реальное поведение)
- Финальные политики могут **различаться** около опасных клеток

In [ ]:
# ============================================================================
# ЭКСПЕРИМЕНТ A: SARSA vs Q-learning на стандартной FrozenLake
# ============================================================================
#
# TODO: заполните пропуски (...) и запустите ячейку
# ============================================================================

# Шаг 1: Сбрасываем сиды для воспроизводимости результатов
np.random.seed(SEED)
random.seed(SEED)

# Шаг 2: Создаём конфигурации алгоритмов (используем значения по умолчанию)
sarsa_config = SARSAConfig()
ql_config = QLearningConfig()

print("Параметры обучения:")
print(f"  Эпизодов: {sarsa_config.num_episodes}")
print(f"  γ (gamma): {sarsa_config.gamma}")
print(f"  α (alpha): {sarsa_config.alpha}")
print(f"  ε: {sarsa_config.epsilon_start} → {sarsa_config.epsilon_end}")
print()

# ============================================================================
# Шаг 3: Обучение SARSA
# ============================================================================
print("=" * 50)
print("Обучение SARSA...")
print("=" * 50)

# Создаём стандартную среду (без штрафа за дыры)
env_std = make_env(hole_penalty=0.0, is_slippery=False)

# Запускаем обучение SARSA
# TODO: вызовите функцию sarsa() с правильными аргументами
Q_sarsa_std, rewards_sarsa_std, success_sarsa_std = ...  # sarsa(env_std, sarsa_config)

env_std.close()

# Выводим результат
print(f"\nSARSA — финальный success rate: {success_sarsa_std[-1] * 100:.1f}%")

# ============================================================================
# Шаг 4: Обучение Q-learning
# ============================================================================
print("\n" + "=" * 50)
print("Обучение Q-learning...")
print("=" * 50)

# Сбрасываем сиды для честного сравнения
np.random.seed(SEED)
random.seed(SEED)

# Создаём такую же среду
env_std = make_env(hole_penalty=0.0, is_slippery=False)

# Запускаем обучение Q-learning
# TODO: вызовите функцию q_learning() с правильными аргументами
Q_ql_std, rewards_ql_std, success_ql_std = ...  # q_learning(env_std, ql_config)

env_std.close()

# Выводим результат
print(f"\nQ-learning — финальный success rate: {success_ql_std[-1] * 100:.1f}%")

# ============================================================================
# Шаг 5: Сравнение политик
# ============================================================================
print("\n" + "=" * 50)
print("Сравнение выученных политик")
print("=" * 50)

# Действия: 0=Left(←), 1=Down(↓), 2=Right(→), 3=Up(↑)
action_symbols = ['←', '↓', '→', '↑']

print("\nПолитика SARSA (оптимальное действие для каждого состояния):")
sarsa_policy = np.argmax(Q_sarsa_std, axis=1).reshape(4, 4)
print(sarsa_policy)

print("\nПолитика Q-learning:")
ql_policy = np.argmax(Q_ql_std, axis=1).reshape(4, 4)
print(ql_policy)

# Проверяем, совпадают ли политики
if np.array_equal(sarsa_policy, ql_policy):
    print("\n✓ Политики ИДЕНТИЧНЫ")
else:
    diff_count = np.sum(sarsa_policy != ql_policy)
    print(f"\n✗ Политики РАЗЛИЧАЮТСЯ в {diff_count} состояниях")

In [ ]:
# ============================================================================
# ВИЗУАЛИЗАЦИЯ ЭКСПЕРИМЕНТА A
# ============================================================================
#
# TODO: запустите эту ячейку после успешного выполнения предыдущей
# ============================================================================

# Ось X: номера эпизодов (каждая точка = среднее за WINDOW эпизодов)
episode_axis = np.arange(len(rewards_sarsa_std)) * WINDOW

# Создаём фигуру с двумя графиками
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ============================================================================
# График 1: Средняя награда за эпизод
# ============================================================================
axes[0].plot(episode_axis, rewards_sarsa_std, label='SARSA', linewidth=2, color='blue')
axes[0].plot(episode_axis, rewards_ql_std, label='Q-learning', linewidth=2, color='orange')
axes[0].set_title('Эксперимент A: Средняя награда', fontsize=14)
axes[0].set_xlabel('Эпизоды')
axes[0].set_ylabel(f'Средняя награда (окно {WINDOW})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ============================================================================
# График 2: Success Rate (доля успешных эпизодов)
# ============================================================================
axes[1].plot(episode_axis, np.array(success_sarsa_std) * 100, 
             label='SARSA', linewidth=2, color='blue')
axes[1].plot(episode_axis, np.array(success_ql_std) * 100, 
             label='Q-learning', linewidth=2, color='orange')
axes[1].set_title('Эксперимент A: Success Rate', fontsize=14)
axes[1].set_xlabel('Эпизоды')
axes[1].set_ylabel('Успешные эпизоды, %')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 105)  # Ограничиваем ось Y для наглядности

plt.tight_layout()
plt.show()

# Выводим финальные метрики
print("\n" + "=" * 50)
print("ИТОГИ ЭКСПЕРИМЕНТА A")
print("=" * 50)
print(f"SARSA      — финальный success rate: {success_sarsa_std[-1] * 100:.1f}%")
print(f"Q-learning — финальный success rate: {success_ql_std[-1] * 100:.1f}%")

### Выводы по эксперименту A
- TODO: Какой алгоритм достиг лучшей финальной производительности?
- TODO: Наблюдаете ли вы разницу в скорости сходимости?
- TODO: Как различаются выученные политики? (проверьте `np.argmax(Q_sarsa, axis=1)` vs `np.argmax(Q_ql, axis=1)`)

## 6. Эксперимент B: Влияние отрицательной награды за дыры

Теперь используем модифицированную среду с `hole_penalty=-1.0` и сравним поведение алгоритмов.

### Зачем нужен штраф за дыры?
В стандартной FrozenLake награда за падение в дыру = 0 (такая же, как за обычный шаг).
Агент не "боится" дыр — он просто не получает положительную награду.

Добавляя **отрицательную награду** за дыры:
- Агент начинает **избегать** опасных клеток
- SARSA (on-policy) учитывает реальные падения при исследовании
- Q-learning (off-policy) оценивает только оптимальное поведение

### Гипотеза
| Алгоритм | Ожидаемое поведение |
|----------|---------------------|
| **SARSA** | Более **консервативный** — избегает рискованных путей, т.к. учитывает ε-случайные падения |
| **Q-learning** | Более **агрессивный** — выбирает короткие маршруты, игнорируя риск исследования |

### План эксперимента
1. Создать среду с `hole_penalty=-1.0`
2. Обучить оба алгоритма
3. Сравнить:
   - Среднюю награду (будет ниже из-за штрафов)
   - Success rate
   - Политики (SARSA должен обходить дыры дальше)

In [ ]:
# ============================================================================
# ЭКСПЕРИМЕНТ B: Влияние штрафа за падение в дыры
# ============================================================================
#
# TODO: заполните пропуски (...) и запустите ячейку
# ============================================================================

print("=" * 50)
print("ЭКСПЕРИМЕНТ B: Среда со штрафом за дыры")
print("=" * 50)
print(f"hole_penalty = -1.0 (стандартно: 0.0)")
print()

# ============================================================================
# Шаг 1: Обучение SARSA на модифицированной среде
# ============================================================================
print("Обучение SARSA + penalty...")

# Создаём среду со штрафом за дыры
# TODO: создайте среду с hole_penalty=-1.0
env_penalty = make_env(hole_penalty=..., is_slippery=False)  # -1.0

# Запускаем обучение (используем ту же конфигурацию)
Q_sarsa_penalty, rewards_sarsa_penalty, success_sarsa_penalty = sarsa(env_penalty, sarsa_config)
env_penalty.close()

print(f"SARSA + penalty — финальный success rate: {success_sarsa_penalty[-1] * 100:.1f}%")

# ============================================================================
# Шаг 2: Обучение Q-learning на модифицированной среде
# ============================================================================
print("\nОбучение Q-learning + penalty...")

# Создаём такую же среду
env_penalty = make_env(hole_penalty=-1.0, is_slippery=False)

# Запускаем обучение
Q_ql_penalty, rewards_ql_penalty, success_ql_penalty = q_learning(env_penalty, ql_config)
env_penalty.close()

print(f"Q-learning + penalty — финальный success rate: {success_ql_penalty[-1] * 100:.1f}%")

# ============================================================================
# Шаг 3: Сравнение со стандартной средой
# ============================================================================
print("\n" + "=" * 50)
print("СРАВНЕНИЕ: стандартная среда vs penalty")
print("=" * 50)
print(f"{'Алгоритм':<20} | {'Стандарт':>10} | {'+ Penalty':>10}")
print("-" * 45)
print(f"{'SARSA':<20} | {success_sarsa_std[-1]*100:>9.1f}% | {success_sarsa_penalty[-1]*100:>9.1f}%")
print(f"{'Q-learning':<20} | {success_ql_std[-1]*100:>9.1f}% | {success_ql_penalty[-1]*100:>9.1f}%")

In [ ]:
# ============================================================================
# ВИЗУАЛИЗАЦИЯ ЭКСПЕРИМЕНТА B: Сравнение 4 вариантов (2×2 сетка)
# ============================================================================
#
# Строки: стандартная среда / penalty среда
# Колонки: средняя награда / success rate
# ============================================================================

episode_axis = np.arange(len(rewards_sarsa_std)) * WINDOW

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ============================================================================
# Строка 1: Стандартная среда (эксперимент A)
# ============================================================================
axes[0, 0].plot(episode_axis, rewards_sarsa_std, label='SARSA', linewidth=2)
axes[0, 0].plot(episode_axis, rewards_ql_std, label='Q-learning', linewidth=2)
axes[0, 0].set_title('Стандартная среда — Награда', fontsize=12)
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(episode_axis, np.array(success_sarsa_std) * 100, label='SARSA', linewidth=2)
axes[0, 1].plot(episode_axis, np.array(success_ql_std) * 100, label='Q-learning', linewidth=2)
axes[0, 1].set_title('Стандартная среда — Success Rate', fontsize=12)
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_ylim(0, 105)

# ============================================================================
# Строка 2: Среда со штрафом (эксперимент B)
# ============================================================================
axes[1, 0].plot(episode_axis, rewards_sarsa_penalty, label='SARSA + penalty', linewidth=2)
axes[1, 0].plot(episode_axis, rewards_ql_penalty, label='Q-learning + penalty', linewidth=2)
axes[1, 0].set_title('Penalty среда — Награда', fontsize=12)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)
# Обратите внимание: награда может быть отрицательной!

axes[1, 1].plot(episode_axis, np.array(success_sarsa_penalty) * 100, label='SARSA + penalty', linewidth=2)
axes[1, 1].plot(episode_axis, np.array(success_ql_penalty) * 100, label='Q-learning + penalty', linewidth=2)
axes[1, 1].set_title('Penalty среда — Success Rate', fontsize=12)
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_ylim(0, 105)

# Подписи осей
for ax in axes.flat:
    ax.set_xlabel('Эпизоды')
for ax in axes[:, 0]:
    ax.set_ylabel('Средняя награда')
for ax in axes[:, 1]:
    ax.set_ylabel('Success Rate, %')

plt.suptitle('Эксперимент B: Влияние штрафа за дыры', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# ВИЗУАЛИЗАЦИЯ ПОЛИТИК (карты оптимальных действий)
# ============================================================================
#
# Показываем, какое действие выбирает агент в каждом состоянии
# ============================================================================

def visualize_policy(Q: np.ndarray, title: str, ax=None) -> None:
    """Отображает оптимальное действие для каждой клетки.
    
    Карта FrozenLake 4×4:
        S F F F      S = Start (старт, состояние 0)
        F H F H      F = Frozen (лёд)
        F F F H      H = Hole (дыра)
        H F F G      G = Goal (цель, состояние 15)
    """
    actions = ['←', '↓', '→', '↑']  # 0=Left, 1=Down, 2=Right, 3=Up
    
    # Извлекаем политику: для каждого состояния берём действие с max Q
    policy = np.argmax(Q, axis=1).reshape(4, 4)
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(4, 4))
    
    # Рисуем стрелки для каждой клетки
    for i in range(4):
        for j in range(4):
            state = i * 4 + j
            ax.text(j, i, actions[policy[i, j]], 
                   ha='center', va='center', fontsize=20, fontweight='bold')
    
    # Настройка осей
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    ax.set_xlim(-0.5, 3.5)
    ax.set_ylim(-0.5, 3.5)
    ax.grid(True, linewidth=2)
    ax.set_title(title, fontsize=11)
    ax.invert_yaxis()  # Чтобы (0,0) было сверху слева


# Создаём сетку 2×2 для всех 4 политик
fig, axes = plt.subplots(2, 2, figsize=(10, 10))

visualize_policy(Q_sarsa_std, 'SARSA — стандартная среда', axes[0, 0])
visualize_policy(Q_ql_std, 'Q-learning — стандартная среда', axes[0, 1])
visualize_policy(Q_sarsa_penalty, 'SARSA — penalty среда', axes[1, 0])
visualize_policy(Q_ql_penalty, 'Q-learning — penalty среда', axes[1, 1])

plt.suptitle('Сравнение политик: SARSA vs Q-learning', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# ============================================================================
# Анализ различий в политиках
# ============================================================================
print("\n" + "=" * 50)
print("АНАЛИЗ ПОЛИТИК")
print("=" * 50)

# Сравниваем политики попарно
policies = {
    'SARSA std': np.argmax(Q_sarsa_std, axis=1),
    'Q-learning std': np.argmax(Q_ql_std, axis=1),
    'SARSA penalty': np.argmax(Q_sarsa_penalty, axis=1),
    'Q-learning penalty': np.argmax(Q_ql_penalty, axis=1)
}

print("\nРазличия между политиками (количество состояний с разными действиями):")
names = list(policies.keys())
for i in range(len(names)):
    for j in range(i+1, len(names)):
        diff = np.sum(policies[names[i]] != policies[names[j]])
        print(f"  {names[i]} vs {names[j]}: {diff} различий")

### Выводы по эксперименту B
- TODO: Как изменилось поведение SARSA при добавлении penalty?
- TODO: Как изменилось поведение Q-learning?
- TODO: Какой алгоритм показал более безопасное поведение?
- TODO: Объясните разницу в средних наградах между алгоритмами

## 7. Эксперимент C: Softmax vs ε-greedy политика

Исследуем влияние **типа политики исследования** на производительность SARSA.

### Два подхода к исследованию

| Политика | Описание | Формула |
|----------|----------|---------|
| **ε-greedy** | С вероятностью ε — случайное действие, иначе — лучшее | $P(a) = \begin{cases} 1-\varepsilon + \varepsilon/n & \text{если } a = \arg\max Q \\ \varepsilon/n & \text{иначе} \end{cases}$ |
| **Softmax** | Вероятности пропорциональны exp(Q/τ) | $P(a) = \frac{e^{Q(s,a)/\tau}}{\sum_{a'} e^{Q(s,a')/\tau}}$ |

### Влияние температуры τ в Softmax

| Температура | Поведение |
|-------------|-----------|
| τ → 0 | Почти жадный выбор (argmax) — мало исследования |
| τ = 1.0 | Сбалансированное поведение |
| τ → ∞ | Почти равномерный выбор — много исследования |

### План эксперимента
1. Обучить SARSA с **ε-greedy** (decay: 1.0 → 0.01) — уже сделано в эксперименте A
2. Обучить SARSA с **Softmax** при разных температурах: τ = 0.5, 1.0, 2.0
3. Сравнить скорость сходимости и финальный success rate

### Гипотеза
- **τ = 0.5** (низкая): быстрая сходимость, но риск застрять в локальном оптимуме
- **τ = 1.0** (средняя): хороший баланс exploration/exploitation
- **τ = 2.0** (высокая): много случайности, медленная сходимость

In [ ]:
# ============================================================================
# ЭКСПЕРИМЕНТ C: Сравнение политик исследования (ε-greedy vs Softmax)
# ============================================================================
#
# TODO: заполните пропуски (...) и запустите ячейку
# ============================================================================

from typing import Dict, List

print("=" * 50)
print("ЭКСПЕРИМЕНТ C: ε-greedy vs Softmax")
print("=" * 50)

# Температуры для тестирования
temperatures = [0.5, 1.0, 2.0]

# Словарь для хранения результатов
softmax_results: Dict[float, Dict] = {}

# ============================================================================
# Обучение SARSA с разными температурами Softmax
# ============================================================================
for temp in temperatures:
    print(f"\n--- Обучение SARSA с Softmax (τ={temp}) ---")
    
    # Создаём среду
    env = make_env(hole_penalty=0.0, is_slippery=False)
    
    # Обучаем SARSA с softmax политикой
    # TODO: вызовите sarsa() с параметрами use_softmax=True и temperature=temp
    Q_soft, rewards_soft, success_soft = sarsa(
        env, 
        sarsa_config, 
        use_softmax=...,      # TODO: True
        temperature=...       # TODO: temp
    )
    env.close()
    
    # Сохраняем результаты
    softmax_results[temp] = {
        'Q': Q_soft,
        'rewards': rewards_soft,
        'success': success_soft
    }
    
    print(f"Softmax τ={temp}: финальный success rate = {success_soft[-1] * 100:.1f}%")

# ============================================================================
# Сравнение с ε-greedy (уже обучен в эксперименте A)
# ============================================================================
print("\n" + "=" * 50)
print("ИТОГИ ЭКСПЕРИМЕНТА C")
print("=" * 50)

print(f"\n{'Политика':<25} | {'Финальный Success Rate':>20}")
print("-" * 50)
print(f"{'ε-greedy (decay)':<25} | {success_sarsa_std[-1] * 100:>19.1f}%")

for temp in temperatures:
    sr = softmax_results[temp]['success'][-1] * 100
    print(f"{'Softmax τ=' + str(temp):<25} | {sr:>19.1f}%")

In [ ]:
# ============================================================================
# ВИЗУАЛИЗАЦИЯ ЭКСПЕРИМЕНТА C: Сравнение политик исследования
# ============================================================================

episode_axis = np.arange(len(success_sarsa_std)) * WINDOW

plt.figure(figsize=(12, 6))

# ε-greedy (baseline из эксперимента A)
plt.plot(episode_axis, np.array(success_sarsa_std) * 100, 
         label='ε-greedy (decay 1.0→0.01)', linewidth=2.5, color='black', linestyle='--')

# Softmax с разными температурами
colors = ['red', 'green', 'blue']
for temp, color in zip(temperatures, colors):
    sr = np.array(softmax_results[temp]['success']) * 100
    plt.plot(episode_axis, sr, label=f'Softmax τ={temp}', linewidth=2, color=color)

plt.title('Эксперимент C: Влияние политики исследования на обучение SARSA', fontsize=14)
plt.xlabel('Эпизоды')
plt.ylabel('Success Rate, %')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.ylim(0, 105)

plt.tight_layout()
plt.show()

# ============================================================================
# Анализ: какая политика лучше?
# ============================================================================
print("\n" + "=" * 50)
print("АНАЛИЗ РЕЗУЛЬТАТОВ")
print("=" * 50)

# Находим лучшую политику
all_results = {'ε-greedy': success_sarsa_std[-1]}
for temp in temperatures:
    all_results[f'Softmax τ={temp}'] = softmax_results[temp]['success'][-1]

best_policy = max(all_results, key=all_results.get)
print(f"\nЛучшая политика: {best_policy} ({all_results[best_policy]*100:.1f}%)")

# Скорость сходимости (когда success rate впервые превысил 50%)
print("\nСкорость сходимости (эпизод, когда SR > 50%):")
for name, sr_list in [('ε-greedy', success_sarsa_std)] + \
                     [(f'Softmax τ={t}', softmax_results[t]['success']) for t in temperatures]:
    sr_array = np.array(sr_list)
    idx = np.where(sr_array > 0.5)[0]
    if len(idx) > 0:
        episode = idx[0] * WINDOW
        print(f"  {name:<20}: эпизод {episode}")
    else:
        print(f"  {name:<20}: не достиг 50%")

### Выводы по эксперименту C
- TODO: Какая температура softmax показала лучший результат?
- TODO: Как температура влияет на баланс исследования/эксплуатации?
- TODO: В каких случаях softmax предпочтительнее ε-greedy?

**Ожидаемые наблюдения:**
- Низкая температура (τ=0.5): более жадное поведение, может застрять в локальном оптимуме
- Средняя температура (τ=1.0): хороший баланс, близко к ε-greedy
- Высокая температура (τ=2.0): слишком много исследования, медленная сходимость
- Softmax более плавно переходит от исследования к эксплуатации


## 8. Дополнительный анализ (опционально)

**Задачи для углублённого изучения:**

1. **Анализ траекторий**: запишите несколько эпизодов обученных политик и визуализируйте маршруты
2. **Матрица посещений**: постройте heatmap частоты посещений состояний для разных алгоритмов
3. **Чувствительность к α**: исследуйте влияние learning rate на сходимость
4. **Double Q-learning**: реализуйте и сравните с обычным Q-learning
5. **Анализ Q-значений**: визуализируйте разницу в Q(s,a) между SARSA и Q-learning


In [ ]:
# Дополнительный анализ: оценка greedy-политик
#
# TODO (опционально): раскомментируйте для финальной проверки

# def evaluate_greedy_policy(Q: np.ndarray, hole_penalty: float = 0.0, episodes: int = 200) -> Tuple[int, int]:
#     """Оценивает жадную политику на основе Q-таблицы.
#     
#     Returns:
#         (hole_hits, goal_hits): количество падений в дыру и достижений цели
#     """
#     env = make_env(hole_penalty=hole_penalty)
#     hole_hits = 0
#     goal_hits = 0
#     
#     for ep in range(episodes):
#         state, _ = env.reset(seed=SEED + 9000 + ep)
#         for _ in range(200):
#             action = int(np.argmax(Q[state]))  # Жадное действие
#             state, reward, terminated, truncated, _ = env.step(action)
#             if terminated:
#                 if reward > 0:
#                     goal_hits += 1
#                 elif reward < 0:
#                     hole_hits += 1
#                 break
#             if truncated:
#                 break
#     
#     env.close()
#     return hole_hits, goal_hits

# # Оценка всех 4 политик
# table = [
#     ('SARSA std', *evaluate_greedy_policy(Q_sarsa_std, 0.0)),
#     ('Q-learning std', *evaluate_greedy_policy(Q_ql_std, 0.0)),
#     ('SARSA penalty', *evaluate_greedy_policy(Q_sarsa_penalty, -1.0)),
#     ('Q-learning penalty', *evaluate_greedy_policy(Q_ql_penalty, -1.0)),
# ]

# print('Оценка greedy-политик (200 эпизодов):')
# print(f"{'Алгоритм':18s} | {'Цели':>5s} | {'Дыры':>5s}")
# print('-' * 35)
# for name, holes, goals in table:
#     print(f"{name:18s} | {goals:5d} | {holes:5d}")

pass

## 9. Вопросы для самопроверки

1. **TODO:** Почему SARSA называется on-policy, а Q-learning — off-policy? Объясните на примере.

2. **TODO:** В каких ситуациях SARSA предпочтительнее Q-learning? Приведите примеры.

3. **TODO:** Как отрицательная награда за падение в дыру влияет на поведение алгоритмов?

4. **TODO:** Почему softmax с низкой температурой может привести к худшим результатам?

5. **TODO:** Что произойдёт, если установить α=1.0? Будет ли алгоритм сходиться?